# Model Hubs: Storing, Download, Upload

This notebook is the source of truth for the model-hub experiments used by the thesis. It defines the modern checkpoint suite, estimates storage footprints, prepares a download plan, computes exponent-stream compression metrics directly from local `.safetensors` shards, and prepares upload manifests for generated artifacts.

The notebook is CPU-runnable by default. Large model downloads and result uploads are disabled unless the flags below are changed explicitly.


In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import heapq
import json
import math
import os
import shutil
import time
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

CWD = Path.cwd().resolve()
if (CWD / "examples" / "1-model-hubs.ipynb").exists():
    REPO_ROOT = CWD
elif CWD.name == "examples" and (CWD / "1-model-hubs.ipynb").exists():
    REPO_ROOT = CWD.parent
else:
    candidates = [CWD, *CWD.parents]
    REPO_ROOT = next(
        (candidate for candidate in candidates if (candidate / "examples" / "1-model-hubs.ipynb").exists()),
        CWD,
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
MODEL_STORE = EXAMPLES_DIR / "models"
OUTPUT_ROOT = EXAMPLES_DIR / "outputs" / "model_hubs"
RUN_ID = os.environ.get("NEURALZIP_MODEL_HUBS_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_STORE.mkdir(parents=True, exist_ok=True)

ALLOW_MISSING = True
EXECUTE_DOWNLOADS = False
EXECUTE_UPLOAD = False
RUN_SYNTHETIC_SMOKE = True
NGRAM_ORDER = 3
MAX_SHARDS_PER_MODEL = None
MAX_TENSORS_PER_MODEL = None

print(f"repo_root={REPO_ROOT}")
print(f"run_dir={RUN_DIR}")


repo_root=/Users/martin/Projects/Tesis/Codebase
run_dir=/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z


## Modern model suite

The suite is defined here rather than imported from project scripts. Parameter counts are rounded metadata used for storage planning; the experiment cells below read the actual local checkpoint shards when they are available.


In [2]:
MODEL_SUITE = [
    {
        "name": "Llama 3.3 70B Instruct",
        "model_id": "meta-llama/Llama-3.3-70B-Instruct",
        "parameters_b": 70.0,
        "default_dtype": "bf16",
        "access": "gated",
        "local_subdir": "meta-llama__Llama-3.3-70B-Instruct",
    },
    {
        "name": "Gemma 4 31B",
        "model_id": "google/gemma-4-31B",
        "parameters_b": 31.0,
        "default_dtype": "bf16",
        "access": "gated-or-token",
        "local_subdir": "google__gemma-4-31B",
    },
    {
        "name": "gpt-oss-120b",
        "model_id": "openai/gpt-oss-120b",
        "parameters_b": 120.0,
        "default_dtype": "bf16",
        "access": "token-if-required",
        "local_subdir": "openai__gpt-oss-120b",
    },
    {
        "name": "Qwen2.5 72B Instruct",
        "model_id": "Qwen/Qwen2.5-72B-Instruct",
        "parameters_b": 72.0,
        "default_dtype": "bf16",
        "access": "open-or-token",
        "local_subdir": "Qwen__Qwen2.5-72B-Instruct",
    },
]

suite_df = pd.DataFrame(MODEL_SUITE)
suite_df["local_dir"] = suite_df["local_subdir"].map(lambda name: str(MODEL_STORE / name))
display(suite_df[["name", "model_id", "parameters_b", "default_dtype", "access", "local_dir"]])


,name,model_id,parameters_b,default_dtype,access,local_dir
0,Llama 3.3 70B Instruct,meta-llama/Llama-3.3-70B-Instruct,70.0,bf16,gated,/Users/martin/Projects/Tesis/Codebase/examples...
1,Gemma 4 31B,google/gemma-4-31B,31.0,bf16,gated-or-token,/Users/martin/Projects/Tesis/Codebase/examples...
2,gpt-oss-120b,openai/gpt-oss-120b,120.0,bf16,token-if-required,/Users/martin/Projects/Tesis/Codebase/examples...
3,Qwen2.5 72B Instruct,Qwen/Qwen2.5-72B-Instruct,72.0,bf16,open-or-token,/Users/martin/Projects/Tesis/Codebase/examples...


## Model storing analysis

For a checkpoint with \(P\) parameters and \(b\) bits per parameter, the tensor payload has \(Pb\) bits. The table below estimates payload bytes and a lightweight expected checkpoint footprint that reserves a small overhead for shard indexes, tensor names, tokenizer/config files, and serialization metadata.


In [3]:
DTYPE_BITS = {"bf16": 16, "fp16": 16, "fp32": 32}
CHECKPOINT_OVERHEAD_FACTOR = 1.02
BYTES_PER_GIB = 1024**3

storage_rows = []
for model in MODEL_SUITE:
    parameter_count = int(model["parameters_b"] * 1_000_000_000)
    for dtype, bits_per_parameter in DTYPE_BITS.items():
        parameter_bits = parameter_count * bits_per_parameter
        storage_bytes = parameter_bits // 8
        expected_checkpoint_bytes = int(storage_bytes * CHECKPOINT_OVERHEAD_FACTOR)
        storage_rows.append(
            {
                "model": model["name"],
                "model_id": model["model_id"],
                "parameters_b": model["parameters_b"],
                "dtype": dtype,
                "bits_per_parameter": bits_per_parameter,
                "parameter_bits": parameter_bits,
                "storage_bytes": storage_bytes,
                "storage_gib": storage_bytes / BYTES_PER_GIB,
                "checkpoint_overhead_factor": CHECKPOINT_OVERHEAD_FACTOR,
                "expected_checkpoint_bytes": expected_checkpoint_bytes,
                "expected_checkpoint_gib": expected_checkpoint_bytes / BYTES_PER_GIB,
            }
        )

storage_table = pd.DataFrame(storage_rows)
storage_table.to_csv(RUN_DIR / "storage_estimates.csv", index=False)
display(
    storage_table[[
        "model", "dtype", "parameters_b", "bits_per_parameter", "storage_gib", "expected_checkpoint_gib"
    ]].round({"storage_gib": 2, "expected_checkpoint_gib": 2})
)


,model,dtype,parameters_b,bits_per_parameter,storage_gib,expected_checkpoint_gib
0,Llama 3.3 70B Instruct,bf16,70.0,16,130.39,132.99
1,Llama 3.3 70B Instruct,fp16,70.0,16,130.39,132.99
2,Llama 3.3 70B Instruct,fp32,70.0,32,260.77,265.99
3,Gemma 4 31B,bf16,31.0,16,57.74,58.90
4,Gemma 4 31B,fp16,31.0,16,57.74,58.90
5,Gemma 4 31B,fp32,31.0,32,115.48,117.79
6,gpt-oss-120b,bf16,120.0,16,223.52,227.99
7,gpt-oss-120b,fp16,120.0,16,223.52,227.99
8,gpt-oss-120b,fp32,120.0,32,447.03,455.98
9,Qwen2.5 72B Instruct,bf16,72.0,16,134.11,136.79


## Download pipeline

Downloads are dry-run by default. Set `EXECUTE_DOWNLOADS = True` only when the machine has enough disk space and the required Hugging Face access token. The notebook calls `snapshot_download` only inside the enabled branch and always writes a `download_manifest` into this run directory.


In [4]:
DOWNLOAD_ALLOW_PATTERNS = [
    "*.safetensors",
    "*.json",
    "*.model",
    "*.txt",
    "*.md",
    "tokenizer*",
    "generation_config.json",
    "special_tokens_map.json",
]
DOWNLOAD_IGNORE_PATTERNS = ["*.bin", "*.pt", "*.onnx", "*.msgpack", "original/*"]
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

download_events = []
for model in MODEL_SUITE:
    local_dir = MODEL_STORE / model["local_subdir"]
    event = {
        "model": model["name"],
        "model_id": model["model_id"],
        "local_dir": str(local_dir),
        "execute": EXECUTE_DOWNLOADS,
        "status": "planned",
        "allow_patterns": DOWNLOAD_ALLOW_PATTERNS,
        "ignore_patterns": DOWNLOAD_IGNORE_PATTERNS,
    }
    if EXECUTE_DOWNLOADS:
        from huggingface_hub import snapshot_download

        try:
            snapshot_path = snapshot_download(
                repo_id=model["model_id"],
                local_dir=str(local_dir),
                allow_patterns=DOWNLOAD_ALLOW_PATTERNS,
                ignore_patterns=DOWNLOAD_IGNORE_PATTERNS,
                token=HF_TOKEN,
            )
            event.update({"status": "downloaded", "snapshot_path": snapshot_path})
        except Exception as exc:
            if not ALLOW_MISSING:
                raise
            event.update({"status": "error", "error": repr(exc)})
    else:
        event.update({"status": "dry_run"})
    download_events.append(event)

download_manifest = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "execute_downloads": EXECUTE_DOWNLOADS,
    "model_store": str(MODEL_STORE),
    "events": download_events,
}
DOWNLOAD_MANIFEST_PATH = RUN_DIR / "download_manifest.json"
DOWNLOAD_MANIFEST_PATH.write_text(json.dumps(download_manifest, indent=2), encoding="utf-8")

download_plan = pd.DataFrame(download_events)
display(download_plan[["model", "model_id", "status", "local_dir"]])
print(f"download_manifest={DOWNLOAD_MANIFEST_PATH}")


,model,model_id,status,local_dir
0,Llama 3.3 70B Instruct,meta-llama/Llama-3.3-70B-Instruct,dry_run,/Users/martin/Projects/Tesis/Codebase/examples...
1,Gemma 4 31B,google/gemma-4-31B,dry_run,/Users/martin/Projects/Tesis/Codebase/examples...
2,gpt-oss-120b,openai/gpt-oss-120b,dry_run,/Users/martin/Projects/Tesis/Codebase/examples...
3,Qwen2.5 72B Instruct,Qwen/Qwen2.5-72B-Instruct,dry_run,/Users/martin/Projects/Tesis/Codebase/examples...


download_manifest=/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/download_manifest.json


## Experiment pipeline

The following cells implement the compression measurement logic in the notebook. They read local `.safetensors` shards with `safe_open`, extract floating-point exponent fields tensor by tensor, and compute three methods:

- **Global Huffman**: one Huffman code over all exponent symbols in the model.
- **ZipNN**: tensor-wise Huffman coding over exponent streams.
- **NeuralZip**: clustered tensor-level `n`-gram Huffman coding with raw tails.


In [5]:
def huffman_bit_cost_from_counts(counts: dict[int, int] | Counter) -> int:
    frequencies = [int(value) for value in counts.values() if int(value) > 0]
    if not frequencies:
        return 0
    if len(frequencies) == 1:
        return frequencies[0]
    heap = frequencies[:]
    heapq.heapify(heap)
    total = 0
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = left + right
        total += merged
        heapq.heappush(heap, merged)
    return int(total)


def counts_from_array(values: np.ndarray) -> Counter:
    if values.size == 0:
        return Counter()
    unique, counts = np.unique(values, return_counts=True)
    return Counter({int(symbol): int(count) for symbol, count in zip(unique, counts)})


def packed_ngram_counts(values: np.ndarray, n: int, exponent_bits: int) -> tuple[Counter, int]:
    if values.size < n:
        return Counter(), int(values.size)
    q = values.size // n
    tail = int(values.size - q * n)
    blocks = values[: q * n].reshape(q, n).astype(np.uint64, copy=False)
    shifts = (np.arange(n, dtype=np.uint64) * np.uint64(exponent_bits))
    packed = np.bitwise_or.reduce(blocks << shifts, axis=1)
    return counts_from_array(packed), tail


def tensor_cluster_key(dtype_name: str, exponent_bits: int, shape: tuple[int, ...], numel: int) -> str:
    size_bin = int(math.floor(math.log2(max(1, int(numel)))))
    return f"dtype={dtype_name}|E={exponent_bits}|rank={len(shape)}|log2_numel={size_bin}"


def safe_ratio(numerator: int | float, denominator: int | float) -> float | None:
    return None if not denominator else float(numerator) / float(denominator)


In [6]:
def tensor_exponent_symbols(tensor: Any) -> tuple[np.ndarray, int, str] | None:
    import torch

    if not torch.is_tensor(tensor) or not tensor.is_floating_point():
        return None

    dtype_name = str(tensor.dtype).replace("torch.", "")
    cpu_tensor = tensor.detach().cpu().contiguous()

    if dtype_name == "bfloat16":
        bits = cpu_tensor.view(torch.int16).numpy().view(np.uint16)
        exponents = ((bits >> 7) & 0xFF).astype(np.uint16, copy=False)
        return exponents.reshape(-1), 8, dtype_name

    if dtype_name == "float16":
        bits = cpu_tensor.view(torch.int16).numpy().view(np.uint16)
        exponents = ((bits >> 10) & 0x1F).astype(np.uint16, copy=False)
        return exponents.reshape(-1), 5, dtype_name

    if dtype_name == "float32":
        bits = cpu_tensor.numpy().view(np.uint32)
        exponents = ((bits >> 23) & 0xFF).astype(np.uint16, copy=False)
        return exponents.reshape(-1), 8, dtype_name

    if dtype_name == "float64":
        bits = cpu_tensor.numpy().view(np.uint64)
        exponents = ((bits >> 52) & 0x7FF).astype(np.uint16, copy=False)
        return exponents.reshape(-1), 11, dtype_name

    return None


In [7]:
def analyze_safetensors_dir(model: dict[str, Any], local_dir: Path) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    try:
        from safetensors import safe_open
    except Exception as exc:
        if not ALLOW_MISSING:
            raise
        return {
            "model": model["name"],
            "model_id": model["model_id"],
            "local_dir": str(local_dir),
            "status": "missing_dependency",
            "error": repr(exc),
            "neuralzip_clusters": None,
            "ngram_order": NGRAM_ORDER,
        }, []

    shards = sorted(local_dir.glob("**/*.safetensors")) if local_dir.exists() else []
    if MAX_SHARDS_PER_MODEL is not None:
        shards = shards[:MAX_SHARDS_PER_MODEL]

    if not shards:
        return {
            "model": model["name"],
            "model_id": model["model_id"],
            "local_dir": str(local_dir),
            "status": "missing",
            "safetensor_shards": 0,
            "checkpoint_bytes": 0,
            "raw_exponent_bits": 0,
            "global_huffman_bits": None,
            "zipnn_tensor_huffman_bits": None,
            "neuralzip_clustered_ngram_bits": None,
            "global_huffman_ratio": None,
            "zipnn_ratio": None,
            "neuralzip_ratio": None,
            "neuralzip_clusters": None,
            "ngram_order": NGRAM_ORDER,
        }, []

    started = time.perf_counter()
    global_counts = Counter()
    neuralzip_cluster_counts: dict[str, Counter] = defaultdict(Counter)
    neuralzip_tail_bits: dict[str, int] = defaultdict(int)
    cluster_rows = []

    raw_exponent_bits = 0
    zipnn_tensor_huffman_bits = 0
    tensor_count = 0
    supported_tensor_count = 0
    parameter_count = 0
    dtype_counts = Counter()

    for shard in shards:
        with safe_open(str(shard), framework="pt", device="cpu") as handle:
            for tensor_name in handle.keys():
                if MAX_TENSORS_PER_MODEL is not None and tensor_count >= MAX_TENSORS_PER_MODEL:
                    break
                tensor = handle.get_tensor(tensor_name)
                tensor_count += 1
                extracted = tensor_exponent_symbols(tensor)
                if extracted is None:
                    continue

                exponents, exponent_bits, dtype_name = extracted
                numel = int(exponents.size)
                shape = tuple(int(dim) for dim in tensor.shape)
                parameter_count += numel
                supported_tensor_count += 1
                dtype_counts[dtype_name] += 1
                raw_exponent_bits += numel * exponent_bits

                tensor_counts = counts_from_array(exponents)
                global_counts.update(tensor_counts)
                zipnn_tensor_huffman_bits += huffman_bit_cost_from_counts(tensor_counts)

                cluster_key = tensor_cluster_key(dtype_name, exponent_bits, shape, numel)
                ngram_counts, tail = packed_ngram_counts(exponents, NGRAM_ORDER, exponent_bits)
                neuralzip_cluster_counts[cluster_key].update(ngram_counts)
                neuralzip_tail_bits[cluster_key] += tail * exponent_bits

                cluster_rows.append(
                    {
                        "model": model["name"],
                        "model_id": model["model_id"],
                        "shard": str(shard),
                        "tensor": tensor_name,
                        "shape": "x".join(map(str, shape)),
                        "numel": numel,
                        "dtype": dtype_name,
                        "exponent_bits": exponent_bits,
                        "cluster_key": cluster_key,
                        "ngram_order": NGRAM_ORDER,
                        "raw_exponent_bits": numel * exponent_bits,
                        "zipnn_tensor_huffman_bits": huffman_bit_cost_from_counts(tensor_counts),
                        "ngram_symbols": sum(ngram_counts.values()),
                        "tail_bits": tail * exponent_bits,
                    }
                )
            if MAX_TENSORS_PER_MODEL is not None and tensor_count >= MAX_TENSORS_PER_MODEL:
                break

    global_huffman_bits = huffman_bit_cost_from_counts(global_counts)
    neuralzip_clustered_ngram_bits = int(
        sum(huffman_bit_cost_from_counts(counts) for counts in neuralzip_cluster_counts.values())
        + sum(neuralzip_tail_bits.values())
    )
    checkpoint_bytes = sum(shard.stat().st_size for shard in shards)

    result = {
        "model": model["name"],
        "model_id": model["model_id"],
        "local_dir": str(local_dir),
        "status": "success" if supported_tensor_count else "no_supported_tensors",
        "safetensor_shards": len(shards),
        "checkpoint_bytes": checkpoint_bytes,
        "parameter_count": parameter_count,
        "tensor_count": tensor_count,
        "supported_tensor_count": supported_tensor_count,
        "dtype_counts": dict(dtype_counts),
        "raw_exponent_bits": int(raw_exponent_bits),
        "global_huffman_bits": int(global_huffman_bits),
        "zipnn_tensor_huffman_bits": int(zipnn_tensor_huffman_bits),
        "neuralzip_clustered_ngram_bits": int(neuralzip_clustered_ngram_bits),
        "global_huffman_ratio": safe_ratio(global_huffman_bits, raw_exponent_bits),
        "zipnn_ratio": safe_ratio(zipnn_tensor_huffman_bits, raw_exponent_bits),
        "neuralzip_ratio": safe_ratio(neuralzip_clustered_ngram_bits, raw_exponent_bits),
        "neuralzip_clusters": len(neuralzip_cluster_counts),
        "ngram_order": NGRAM_ORDER,
        "elapsed_s": time.perf_counter() - started,
    }
    return result, cluster_rows


In [8]:
experiment_results = []
cluster_details = []

for model in MODEL_SUITE:
    local_dir = MODEL_STORE / model["local_subdir"]
    result, clusters = analyze_safetensors_dir(model, local_dir)
    experiment_results.append(result)
    cluster_details.extend(clusters)

results_table = pd.DataFrame(experiment_results)
cluster_table = pd.DataFrame(cluster_details)

RESULTS_CSV = RUN_DIR / "results.csv"
RESULTS_JSON = RUN_DIR / "results.json"
CLUSTERS_CSV = RUN_DIR / "neuralzip_clusters.csv"
EXPERIMENT_MANIFEST_PATH = RUN_DIR / "experiment_manifest.json"

results_table.to_csv(RESULTS_CSV, index=False)
RESULTS_JSON.write_text(json.dumps(experiment_results, indent=2), encoding="utf-8")
cluster_table.to_csv(CLUSTERS_CSV, index=False)

experiment_manifest = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "allow_missing": ALLOW_MISSING,
    "ngram_order": NGRAM_ORDER,
    "model_store": str(MODEL_STORE),
    "results_csv": str(RESULTS_CSV),
    "results_json": str(RESULTS_JSON),
    "clusters_csv": str(CLUSTERS_CSV),
    "models": experiment_results,
}
EXPERIMENT_MANIFEST_PATH.write_text(json.dumps(experiment_manifest, indent=2), encoding="utf-8")

display_columns = [
    "model", "status", "safetensor_shards", "checkpoint_bytes", "raw_exponent_bits",
    "global_huffman_ratio", "zipnn_ratio", "neuralzip_ratio", "neuralzip_clusters",
]
display(results_table[display_columns])
print(f"results_csv={RESULTS_CSV}")
print(f"experiment_manifest={EXPERIMENT_MANIFEST_PATH}")


,model,status,safetensor_shards,checkpoint_bytes,raw_exponent_bits,global_huffman_ratio,zipnn_ratio,neuralzip_ratio,neuralzip_clusters
0,Llama 3.3 70B Instruct,missing,0,0,0,None,None,None,None
1,Gemma 4 31B,missing,0,0,0,None,None,None,None
2,gpt-oss-120b,missing,0,0,0,None,None,None,None
3,Qwen2.5 72B Instruct,missing,0,0,0,None,None,None,None


results_csv=/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/results.csv
experiment_manifest=/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/experiment_manifest.json


## Local smoke validation

This cell creates a tiny `.safetensors` checkpoint inside the run directory and sends it through the same analysis functions. It does not enter the modern suite results; it is only a lightweight check that exponent extraction, Huffman costs, and clustered `n`-gram coding execute on CPU.


In [9]:
smoke_result = None
if RUN_SYNTHETIC_SMOKE:
    try:
        import torch
        from safetensors.torch import save_file

        smoke_dir = RUN_DIR / "smoke_checkpoint"
        smoke_dir.mkdir(parents=True, exist_ok=True)
        smoke_path = smoke_dir / "model.safetensors"
        torch.manual_seed(7)
        save_file(
            {
                "layers.0.weight": torch.randn(64, 64, dtype=torch.float32),
                "layers.1.weight": torch.randn(32, 64, dtype=torch.float16),
                "norm.weight": torch.randn(64, dtype=torch.bfloat16),
            },
            str(smoke_path),
        )
        smoke_model = {
            "name": "synthetic safetensors smoke",
            "model_id": "local/synthetic-safetensors-smoke",
            "local_subdir": str(smoke_dir),
        }
        smoke_result, smoke_clusters = analyze_safetensors_dir(smoke_model, smoke_dir)
        (RUN_DIR / "smoke_result.json").write_text(json.dumps(smoke_result, indent=2), encoding="utf-8")
        pd.DataFrame(smoke_clusters).to_csv(RUN_DIR / "smoke_clusters.csv", index=False)
        display(pd.DataFrame([smoke_result]))
    except Exception as exc:
        if not ALLOW_MISSING:
            raise
        smoke_result = {"status": "skipped", "error": repr(exc)}
        (RUN_DIR / "smoke_result.json").write_text(json.dumps(smoke_result, indent=2), encoding="utf-8")
        print(smoke_result)
else:
    smoke_result = {"status": "disabled"}
    print(smoke_result)


,model,model_id,local_dir,status,safetensor_shards,checkpoint_bytes,parameter_count,tensor_count,supported_tensor_count,dtype_counts,raw_exponent_bits,global_huffman_bits,zipnn_tensor_huffman_bits,neuralzip_clustered_ngram_bits,global_huffman_ratio,zipnn_ratio,neuralzip_ratio,neuralzip_clusters,ngram_order,elapsed_s
0,synthetic safetensors smoke,local/synthetic-safetensors-smoke,/Users/martin/Projects/Tesis/Codebase/examples...,success,1,20848,6208,3,3,"{'float32': 1, 'float16': 1, 'bfloat16': 1}",43520,21668,15935,14891,0.497886,0.366153,0.342165,3,3,0.008037


## Upload pipeline

Uploads are dry-run by default. The local provider copies the run directory to another local folder. The Hugging Face provider uses `HfApi.upload_folder` only when `EXECUTE_UPLOAD = True` and `UPLOAD_PROVIDER = "huggingface"`.


In [10]:
UPLOAD_PROVIDER = "local"  # "local" or "huggingface"
LOCAL_UPLOAD_ROOT = OUTPUT_ROOT / "uploaded"
HF_UPLOAD_REPO_ID = os.environ.get("HF_UPLOAD_REPO_ID", "")
HF_UPLOAD_REPO_TYPE = "dataset"
HF_UPLOAD_PATH_IN_REPO = f"model_hubs/{RUN_ID}"

artifact_paths = sorted(path for path in RUN_DIR.glob("**/*") if path.is_file())
upload_event = {
    "run_id": RUN_ID,
    "execute": EXECUTE_UPLOAD,
    "provider": UPLOAD_PROVIDER,
    "artifact_count": len(artifact_paths),
    "artifact_bytes": sum(path.stat().st_size for path in artifact_paths),
    "artifacts": [str(path.relative_to(RUN_DIR)) for path in artifact_paths],
    "status": "dry_run",
}

if EXECUTE_UPLOAD:
    if UPLOAD_PROVIDER == "local":
        destination = LOCAL_UPLOAD_ROOT / RUN_ID
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(RUN_DIR, destination)
        upload_event.update({"status": "copied", "destination": str(destination)})
    elif UPLOAD_PROVIDER == "huggingface":
        if not HF_UPLOAD_REPO_ID:
            raise ValueError("Set HF_UPLOAD_REPO_ID before enabling Hugging Face uploads.")
        from huggingface_hub import HfApi

        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=str(RUN_DIR),
            repo_id=HF_UPLOAD_REPO_ID,
            repo_type=HF_UPLOAD_REPO_TYPE,
            path_in_repo=HF_UPLOAD_PATH_IN_REPO,
        )
        upload_event.update(
            {
                "status": "uploaded",
                "repo_id": HF_UPLOAD_REPO_ID,
                "repo_type": HF_UPLOAD_REPO_TYPE,
                "path_in_repo": HF_UPLOAD_PATH_IN_REPO,
            }
        )
    else:
        raise ValueError(f"Unknown upload provider: {UPLOAD_PROVIDER}")

upload_manifest = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "event": upload_event,
}
UPLOAD_MANIFEST_PATH = RUN_DIR / "upload_manifest.json"
UPLOAD_MANIFEST_PATH.write_text(json.dumps(upload_manifest, indent=2), encoding="utf-8")

display(pd.DataFrame([upload_event]).drop(columns=["artifacts"]))
print(f"upload_manifest={UPLOAD_MANIFEST_PATH}")


,run_id,execute,provider,artifact_count,artifact_bytes,status
0,20260609T025559Z,False,local,9,32960,dry_run


upload_manifest=/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/upload_manifest.json


## Run summary

The run directory contains storage estimates, dry-run download/upload manifests, model-level experiment rows, tensor-cluster details when checkpoints are present, and the synthetic smoke artifacts.


In [11]:
run_summary = {
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "storage_estimates": str(RUN_DIR / "storage_estimates.csv"),
    "download_manifest": str(DOWNLOAD_MANIFEST_PATH),
    "results_csv": str(RESULTS_CSV),
    "experiment_manifest": str(EXPERIMENT_MANIFEST_PATH),
    "upload_manifest": str(UPLOAD_MANIFEST_PATH),
    "allow_missing": ALLOW_MISSING,
    "execute_downloads": EXECUTE_DOWNLOADS,
    "execute_upload": EXECUTE_UPLOAD,
    "smoke_status": smoke_result.get("status") if isinstance(smoke_result, dict) else None,
}
(RUN_DIR / "run_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
run_summary


{'run_id': '20260609T025559Z',
 'run_dir': '/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z',
 'storage_estimates': '/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/storage_estimates.csv',
 'download_manifest': '/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/download_manifest.json',
 'results_csv': '/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/results.csv',
 'experiment_manifest': '/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/experiment_manifest.json',
 'upload_manifest': '/Users/martin/Projects/Tesis/Codebase/examples/outputs/model_hubs/20260609T025559Z/upload_manifest.json',
 'allow_missing': True,
 'execute_downloads': False,
 'execute_upload': False,
 'smoke_status': 'success'}